In [68]:
import pandas as pd
import numpy as np
from pathlib import Path
import altair as alt

In [69]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/"

df = pd.read_csv(root + "MainDataFrame_MitoticStopwatch_Monastrol.csv")

In [70]:
#df = df[df["Dataset"] == 20250627]

In [71]:
colourscheme = "viridis"

In [72]:
def Binned_Circle2(
    data, x, y, x_title, y_title, color, column, 
    Circlesize = 125,  
    Bin_Boxplot_width = 100, 
    Bin_Boxplot_height = 150,
    RawPointSize = 18
                ):

    # Shared encodings
    base_x = alt.X(x, title = None, axis = alt.Axis(grid = False))
    base_color = alt.Color(color, scale = alt.Scale(scheme = colourscheme))

    # Mean Circle
    mean = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = Circlesize).encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Mean Error Bar
    mean_sem = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_errorbar().encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Raw Data Points
    raw_points = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = RawPointSize, opacity = 0.1).encode(
        x = base_x,
        y = alt.Y(y, title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Layer all three
    layered = raw_points + mean_sem + mean

    # Apply facet to combined chart
    faceted = layered.facet(
        column = column,
        spacing = 15
    )

    # Configure chart
    return faceted.configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 12
           ).configure_header(
                labelOrient = 'bottom', title = None
           ).configure_view(
                stroke = 'transparent', 
                strokeWidth = 0.5
           )

In [73]:
def Scatter_bin(dataframe, x, y, color, x_title, y_title, binextent, binstep,
            Circlesize = 15, 
            CircleOpacity = 0.2,  
            Scatter_width = 250, 
            Scatter_height = 200
               ):
    
    # Standard scatter plot 
    Scatter = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = CircleOpacity,
        size = Circlesize
    ).encode(
        alt.X(x, title = x_title),
        alt.Y(y, title = y_title),
        color = alt.Color(
            color, legend = None, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Scatter_binned = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = 1,
        size = 100
    ).encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Error_Scatterbinned = alt.Chart(
            data = dataframe
    ).mark_errorbar(extent = "ci").encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    ) 
    SCATTERBIN = Error_Scatterbinned + Scatter_binned
    return SCATTERBIN

In [74]:
Scatter_MitoticDuration = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Mitotic_duration_mins",
    color = "Splitting_event",
    x_title = "Time (h)",
    y_title = "Mitotic duration (min)",
    binextent = [0, 72],
    binstep = 6
)
Scatter_MitoticDuration

alt.LayerChart(...)

In [75]:
df[df["Mitotic_duration_mins"] > 0].shape

(489, 52)

In [76]:
Bincircle_arrest = Binned_Circle2(
    data = df[df["Cell_Cycle_hours"] > 1],
    x = 'Mother_arrested',
    y = 'Cell_Cycle_hours',
    x_title = '',
    y_title = 'Cell_Cycle_hours',
    color = 'Mother_arrested:O',
    column = 'Splitting_event'
)
Bincircle_arrest

alt.FacetChart(...)

In [77]:
df.groupby("Mother_arrested").Cell_Cycle_hours.mean()

Mother_arrested
000-030 min    32.200000
030-060        27.676415
060-090        29.346970
090-120        39.520833
> 120 min      48.412179
Name: Cell_Cycle_hours, dtype: float64

In [78]:
df.Mother_arrested.value_counts()

Mother_arrested
030-060        159
060-090         77
> 120 min       52
090-120         12
000-030 min      5
Name: count, dtype: int64

In [79]:
def plot_Individual_Daughters(df, barwidth = 2, plotwidth = 700, plotheight = 200):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mother_Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
        y = alt.Y("Mother_Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, 700])),
        color = alt.Color("Cell_Cycle_hours:Q", title = "Daughter cell cycle duration (hours)", scale = alt.Scale(scheme = "brownbluegreen")),
    )

    chart = bar.properties(
        width = plotwidth,
        height = plotheight,
    )

    return chart

In [80]:
# All data without selection!

daughters = plot_Individual_Daughters(df[df["Mother_Mitotic_duration_mins"] > 0], barwidth = 2, plotwidth = 600, plotheight = 250)
daughters

alt.Chart(...)

In [81]:
df.Unique_Track_ID.nunique()

183

In [101]:
df[df["Mother_Mitotic_duration_mins"] > 0].shape

(305, 53)

In [82]:
def compute_generation_percentages(df):
    # Helper to compute percentages by dataset
    def calculate_percentages(sub_df, population_label):
        # Get max generation per Track_ID
        grouped = sub_df.groupby(["Unique_Track_ID", "Dataset"])["Generation"].max().reset_index()

        # Compute category flags
        grouped["Max_Gen1"] = grouped["Generation"] == 1
        grouped["Max_Gen2"] = grouped["Generation"] == 2
        grouped["Max_Gen3"] = grouped["Generation"] == 3
        grouped["Population"] = population_label

        # Aggregate counts and percentages by Dataset
        summary = grouped.groupby(["Dataset", "Population"]).agg(
            Total_Tracks = ("Unique_Track_ID", "count"),
            Tracks_Only_Gen1 = ("Max_Gen1", "sum"),
            Tracks_Only_Gen2 = ("Max_Gen2", "sum"),
            Tracks_Only_Gen3 = ("Max_Gen3", "sum")
        ).reset_index()

        # Calculate percentages
        summary["Max_Gen1_percent"] = 100 * summary["Tracks_Only_Gen1"] / summary["Total_Tracks"]
        summary["Max_Gen2_percent"] = 100 * summary["Tracks_Only_Gen2"] / summary["Total_Tracks"]
        summary["Max_Gen3_percent"] = 100 * summary["Tracks_Only_Gen3"] / summary["Total_Tracks"]

        return summary[[
            "Dataset", "Population", "Total_Tracks",
            "Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"
        ]]

    # Mother cell prolonged mitotic arrest
    gen1_high = df[(df["Generation"] == 1) & (df["Mitotic_duration_mins"] > 100)]
    high_track_ids = gen1_high["Unique_Track_ID"].unique()
    df_high = df[df["Unique_Track_ID"].isin(high_track_ids)]
    summary_high = calculate_percentages(df_high, "High_Mitotic")

    # Mother cell little or no mitotic arrest
    gen1_low = df[(df["Generation"] == 1) & (df["Mitotic_duration_mins"] < 100)]
    low_track_ids = gen1_low["Unique_Track_ID"].unique()
    df_low = df[df["Unique_Track_ID"].isin(low_track_ids)]
    summary_low = calculate_percentages(df_low, "Low_Mitotic")

    # Combine results
    final_summary = pd.concat([summary_high, summary_low], ignore_index = True)

    # Reshape to long format
    final_long = final_summary.melt(
        id_vars = ["Dataset", "Population", "Total_Tracks"],
        value_vars = ["Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"],
        var_name = "Metric",
        value_name = "Percentage"
    )

    return final_long

percentages_df_all = compute_generation_percentages(df)

percentages_df_all.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_all.csv") # Generation percentages of all cells (defects + no defects)

percentages_df_all

,Dataset,Population,Total_Tracks,Metric,Percentage
0,20250627.0,High_Mitotic,43,Max_Gen1_percent,51.162791
1,20250704.0,High_Mitotic,38,Max_Gen1_percent,44.736842
2,20250627.0,Low_Mitotic,32,Max_Gen1_percent,18.750000
3,20250704.0,Low_Mitotic,70,Max_Gen1_percent,25.714286
4,20250627.0,High_Mitotic,43,Max_Gen2_percent,41.860465
5,20250704.0,High_Mitotic,38,Max_Gen2_percent,50.000000
6,20250627.0,Low_Mitotic,32,Max_Gen2_percent,37.500000
7,20250704.0,Low_Mitotic,70,Max_Gen2_percent,37.142857
8,20250627.0,High_Mitotic,43,Max_Gen3_percent,6.976744
9,20250704.0,High_Mitotic,38,Max_Gen3_percent,2.631579


In [83]:
def plot_percentage_comparison(df_long):

    bar = alt.Chart(df_long).mark_bar(width = 15).encode(
        x = alt.X("Population:N", title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y("mean(Percentage):Q", title = "% of cell families"),
        color = alt.Color("Metric:N", title = "Metric", scale = alt.Scale(scheme = "viridis")),
        xOffset = "Metric:N"
    )

    error = alt.Chart(df_long).mark_errorbar().encode(
        x = alt.X("Population:N"),
        y = alt.Y("mean(Percentage):Q", title = "% of cell families"),
        color = alt.Color("Metric:N"),
        xOffset = "Metric:N"
    )

    datasets = alt.Chart(df_long).mark_circle(width = 15).encode(
        x = alt.X("Population:N", title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y("Percentage:Q", title = "% of cell families"),
        color = alt.Color("Dataset:N", title = "Metric", scale = alt.Scale(scheme = "viridis")),
        xOffset = "Metric:N"
    )

    chart = (bar + datasets + error).properties(
        width = 200,
        height = 200,
    )

    return chart

In [84]:
# All data, without selection or filtering

chart_all = plot_percentage_comparison(percentages_df_all)
chart_all

alt.LayerChart(...)

In [85]:
# Cohort selection

# Select cutoff manually
Threshold_manual = 90

# Select cropping for cohort
cohort_min = 0 # hours
cohort_max = 18.1 # hours




def mother_divides_in_selection(x, dataframe = df, max_time = cohort_max):
    # Function to find out if mother divided within selection time (cohort max)
    if x.Generation < 2:
        return False
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        mother_time = mother_row["Frame_hrs"]

        if mother_row["Frame_hrs"].shape[0] == 1:   
            if mother_time.item() < max_time:
                return True
            else:
                return False
        else:
            return False

df["Mother_divides_in_time"] = df.apply(mother_divides_in_selection, axis = 1)


print(df["Unique_Track_ID"].nunique())

print(df[(df["Frame_hrs"] < cohort_max) & (df["Frame_onset_hrs"] > cohort_min)].Unique_Track_ID.nunique())

print(df[(df["Frame_hrs"] < cohort_max) & (df["Frame_onset_hrs"] > cohort_min) & (df["Generation"] == 1)].Unique_Track_ID.nunique())

183
142
142


In [86]:
# copied from augmin analysis

def compute_generation_percentages(df, depletion_time_min = cohort_min, depletion_time_max = cohort_max, threshold = Threshold_manual):
    
    # Helper to compute percentages by dataset
    def calculate_percentages(sub_df, population_label):
        
        # Get max generation per Track_ID
        grouped = sub_df.groupby(["Unique_Track_ID", "Dataset"])["Generation"].max().reset_index()

        # Compute category flags
        grouped["Max_Gen1"] = grouped["Generation"] == 1
        grouped["Max_Gen2"] = grouped["Generation"] == 2
        grouped["Max_Gen3"] = grouped["Generation"] == 3
        grouped["Population"] = population_label

        # Aggregate counts and percentages by Dataset
        summary = grouped.groupby(["Dataset", "Population"]).agg(
            Total_Tracks = ("Unique_Track_ID", "count"),
            Tracks_Only_Gen1 = ("Max_Gen1", "sum"),
            Tracks_Only_Gen2 = ("Max_Gen2", "sum"),
            Tracks_Only_Gen3 = ("Max_Gen3", "sum")
        ).reset_index()

        # Calculate percentages
        summary["Max_Gen1_percent"] = 100 * summary["Tracks_Only_Gen1"] / summary["Total_Tracks"]
        summary["Max_Gen2_percent"] = 100 * summary["Tracks_Only_Gen2"] / summary["Total_Tracks"]
        summary["Max_Gen3_percent"] = 100 * summary["Tracks_Only_Gen3"] / summary["Total_Tracks"]

        return summary[[
            "Dataset", "Population", "Total_Tracks",
            "Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"
        ]]

    print(f"The number of total tracks before selecting: {df["Unique_Track_ID"].nunique()}")
    
    # Mother cell prolonged mitotic arrest
    gen1_high = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] > threshold) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    high_track_ids = gen1_high["Unique_Track_ID"].unique()
    print(f"The number of selected tracks that have HIGH mitotic times: {len(high_track_ids)}")
    df_high = df[df["Unique_Track_ID"].isin(high_track_ids)]
    summary_high = calculate_percentages(df_high, f">{threshold} min") # calculating percentages using the helper function

    # Mother cell little or no mitotic arrest
    gen1_low = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] < threshold) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    low_track_ids = gen1_low["Unique_Track_ID"].unique()
    print(f"The number of selected tracks that have LOW mitotic times: {len(low_track_ids)}")
    df_low = df[df["Unique_Track_ID"].isin(low_track_ids)]
    summary_low = calculate_percentages(df_low, f"<{threshold} min") # calculating percentages using the helper function

    # Combine results
    final_summary = pd.concat([summary_high, summary_low], ignore_index = True)

    # Reshape to long format for plotting
    final_long = final_summary.melt(
        id_vars = ["Dataset", "Population", "Total_Tracks"],
        value_vars = ["Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"],
        var_name = "Metric",
        value_name = "Percentage"
    )

    return df_high, df_low, final_long

results_df_high, results_df_low, percentages_df = compute_generation_percentages(df)
results_df_high_noError, results_df_low_noError, percentages_df_noError = compute_generation_percentages(df[df['any_true'] != True])

percentages_df.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_SelectedCohort.csv")
#percentages_df_noError.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_SelectedCohort_NoErrors.csv")

The number of total tracks before selecting: 183
The number of selected tracks that have HIGH mitotic times: 81
The number of selected tracks that have LOW mitotic times: 61
The number of total tracks before selecting: 171
The number of selected tracks that have HIGH mitotic times: 67
The number of selected tracks that have LOW mitotic times: 56


In [87]:
def plot_percentage_comparison(x = "Population:N", 
                               color = "Dataset:O",
                               color_bar = "Metric:N",
                               offset = "Metric:N", 
                               y = "Percentage", 
                               y_title = "% of cell families", 
                               data = percentages_df
                              ):

    bar = alt.Chart(data).mark_bar(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = -0)),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    error = alt.Chart(data).mark_errorbar(extent = "stderr").encode(
        x = alt.X(x),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar),
        xOffset = offset
    )

    datasets = alt.Chart(data).mark_circle(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y(f"mean({y})", title = y_title),
        color = alt.Color(color, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    chart = (bar + datasets + error).properties(
        width = 200,
        height = 200,
    )

    return chart

In [88]:
chart_monastrol = plot_percentage_comparison(
    data = percentages_df
)

chart_monastrol

alt.LayerChart(...)

In [89]:
# with errors filtered 

chart_monastrol_noerrors = plot_percentage_comparison(
    data = percentages_df_noError
)

chart_monastrol_noerrors

alt.LayerChart(...)

In [90]:
result_df = pd.concat([results_df_high, results_df_low])

result_df = result_df[result_df["Frame_hrs"] < cohort_max] 

result_df.to_csv(root + "Filtered_Monastrol.csv")

In [91]:
## UNCOMMENT !!!!!!!!! #######

# filter out errors

#df_filtered = df[df["any_true"] != True]
#result_df_filtered = result_df[result_df["any_true"] != True]


result_df_filtered = result_df # Delete!!!!!


#result_df_filtered.any_true.value_counts()

In [92]:
# Function to expand each row into two daughter rows
def expand_to_daughters(row):
    mother_id = row["Source_ID"]
    mother_duration = row["Mitotic_duration_mins"]
    n_daughters = row["Number_of_daughters_in_mitosis"]
    
    # Determine proliferation values
    if n_daughters == 0:
        proliferation = [False, False]
    elif n_daughters == 1:
        proliferation = [True, False]
    elif n_daughters == 2:
        proliferation = [True, True]
    else:
        raise ValueError(f"Unexpected number of daughters: {n_daughters}")

    return pd.DataFrame({
        "Mother_ID": [mother_id, mother_id],
        "Mother_Mitotic_duration_mins": [mother_duration, mother_duration],
        "Proliferation": proliferation
    })

# Apply function to each row and concatenate results
df_daughters = pd.concat(result_df_filtered.apply(expand_to_daughters, axis = 1).to_list(), ignore_index = True).reset_index() # Errors are already filtered

In [93]:
# Function to plot Source IDs (daughter IDs represented by duplicated mother IDs) sorted by mitotic duration of mother
# and whether the daughters proliferate or not

def plot_Individual_Daughters_proliferate(df, barwidth, y_max, plotwidth = 375):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "index:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mother_Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
        y = alt.Y("Mother_Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, y_max])), 
        color = alt.Color("Proliferation", title = "Daugher proliferates", scale = alt.Scale(scheme = "magma")),
    )

    chart = bar.properties(
        width = plotwidth,
        height = 200,
    )

    return chart

In [94]:
# daughter proliferation no errors

daughters_all = plot_Individual_Daughters_proliferate(
    df_daughters, 
    barwidth = 1.8, 
    y_max = 700,
    plotwidth = 450
)

daughters_all

alt.Chart(...)

In [95]:
df_daughters.shape

(284, 4)

In [96]:
long_mitoses = df_daughters[df_daughters["Mother_Mitotic_duration_mins"] >= 90]
long_mitoses.shape

(162, 4)

In [97]:
long_mitoses[(long_mitoses["Proliferation"] == False)].shape

(99, 4)

In [98]:
short_mitoses = df_daughters[df_daughters["Mother_Mitotic_duration_mins"] <= 90]
short_mitoses.shape

(122, 4)

In [99]:
short_mitoses[(short_mitoses["Proliferation"] == False)].shape

(37, 4)

In [100]:
daughters_selected = plot_Individual_Daughters2(result_df, plotwidth = 350, barwidth = 2.5)
daughters_selected

NameError: name 'plot_Individual_Daughters2' is not defined

In [ ]:
daughters_selected_no_errors = plot_Individual_Daughters2(result_df_filtered, plotwidth = 350, barwidth = 2.8)
daughters_selected_no_errors

In [ ]:
daughters = plot_Individual_Daughters(
    df[df["Mother_divides_in_time"] == True], 
    barwidth = 2.5, 
    plotwidth = 350
)
daughters

In [ ]:
 df[df["Mother_divides_in_time"] == True].shape

In [ ]:
daughters_noError = plot_Individual_Daughters(
    df_filtered[df_filtered["Mother_divides_in_time"] == True], 
    barwidth = 2.5, 
    plotwidth = 350
)
daughters_noError

In [ ]:
 df_filtered[df_filtered["Mother_divides_in_time"] == True].shape

In [ ]:
def Scatter_bin2(dataframe, x, y, color, x_title, y_title, binextent, binstep,
            Circlesize = 20, 
            CircleOpacity = 0.2,  
            Scatter_width = 150, 
            Scatter_height = 150
               ):
    # Standard scatter plot 
    Scatter = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = CircleOpacity,
        size = Circlesize
    ).encode(
        alt.X(x, title = x_title),
        alt.Y(y, title = y_title),
        color = alt.Color(
            color, legend = None, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Scatter_binned = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = 1,
        size = 100
    ).encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Error_Scatterbinned = alt.Chart(
            data = dataframe
    ).mark_errorbar(extent = "ci").encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    ) 
    SCATTERBIN = Error_Scatterbinned + Scatter_binned
    return SCATTERBIN

In [ ]:
Scatter_MitoticDuration_CC = Scatter_bin2(
    dataframe = df_filtered[df_filtered["Mother_divides_in_time"] == True],
    y = "Cell_Cycle_hours",
    x = "Mother_Mitotic_duration_mins",
    color = "Splitting_event",
    y_title = "Daughter Cell cycle duration (h)",
    x_title = "Mother Mitotic duration (min)",
    binextent = [0, 240],
    binstep = 60
)
Scatter_MitoticDuration_CC

In [ ]:
df.columns

In [ ]:
df_filtered[df_filtered["Mother_divides_in_time"] == True].any_true.value_counts()

In [ ]:
df[df["Mother_divides_in_time"] == True].any_true.value_counts()

In [ ]:
cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Divition", 
    'Citokinesis Failure',
    'Slippage',
    'Mitotic Death'
       ]

df_short = df[cols + ['Dataset', 'Metaphase_arrested']]

# Work on a copy to avoid warnings
#bool_df = df_short.loc[:, cols].copy()

# Convert and cast to nullable boolean
df_short[cols] = df_short.loc[:, cols].apply(lambda col: col.astype("boolean"))

print(df_short.dtypes)

In [ ]:
bool_cols = df_short.select_dtypes(include = bool).columns
grouped = df_short.groupby(['Dataset', 'Metaphase_arrested'])[bool_cols].mean().mul(100)
grouped['Normal'] = 100 - grouped.sum(axis=1)

mean_df = grouped.groupby('Metaphase_arrested').mean()
sem_df  = grouped.groupby('Metaphase_arrested').sem().fillna(0)

plot_df = (
    mean_df.reset_index().melt(id_vars = 'Metaphase_arrested', var_name = 'Category', value_name = 'Mean')
    .merge(
        sem_df.reset_index().melt(id_vars = 'Metaphase_arrested', var_name = 'Category', value_name = 'SEM'),
        on = ['Metaphase_arrested', 'Category']
    )
)

# Stacking order
order = [c for c in mean_df.columns if c != 'Normal'] + ['Normal']
plot_df['Category'] = pd.Categorical(plot_df['Category'], categories = order, ordered = True)

# sort by condition then the category order so cumsum matches visual order
plot_df['order_index'] = plot_df['Category'].cat.codes
plot_df = plot_df.sort_values(['Metaphase_arrested', 'order_index']).reset_index(drop = True)

# compute bottoms & tops (explicit stacking)
plot_df['bottom'] = plot_df.groupby('Metaphase_arrested')['Mean'].cumsum() - plot_df['Mean']
plot_df['top']    = plot_df['bottom'] + plot_df['Mean']

# compute error bar extents relative to each slice
plot_df['err_low']  = plot_df['bottom'] + (plot_df['Mean'] - plot_df['SEM'])
plot_df['err_high'] = plot_df['bottom'] + (plot_df['Mean'] + plot_df['SEM'])

# Optionally clamp lower error to bottom (uncomment if you want that):
# plot_df['err_low'] = plot_df[['err_low', 'bottom']].max(axis=1)

# build charts using precomputed bottoms/tops (no implicit stacking)
bars = alt.Chart(plot_df).mark_bar(size = 25).encode(
    x = alt.X('Metaphase_arrested:N', title = 'Mother cell mitotic duration'),
    y = alt.Y('bottom:Q', title = 'Percentage (%)', scale = alt.Scale(domain = [0, 100])),
    y2 = 'top:Q',
    color = alt.Color('Category:N', sort = order, legend = alt.Legend(title = 'Category')),
    tooltip = ['Metaphase_arrested:N', 'Category:N', alt.Tooltip('Mean:Q', format = '.2f'), alt.Tooltip('SEM:Q', format = '.2f')]
).properties(width = 200, height = 200)

# error rules placed at the same horizontal position; colored by category (no legend duplication)
err = alt.Chart(plot_df).mark_rule(strokeWidth = 2).encode(
    x = 'Metaphase_arrested:N',
    y = 'err_low:Q',
    y2 = 'err_high:Q',
    color = alt.Color('Category:N', sort = order, legend = None),
    tooltip = ['Metaphase_arrested:N', 'Category:N', alt.Tooltip('Mean:Q', format = '.2f'), alt.Tooltip('SEM:Q', format = '.2f')]
)

chart = (bars + err).properties(
    #title='Abnormalities & Normal per Condition (mean ± SEM across datasets)'
)

chart